# Lecture 2: Computational Thinking with Python

## Exercise 2: Chemical Equation Balancer Checker

**Objective:** Write a program that checks if a chemical equation is balanced.

**Problem:** Create a function called `is_balanced()` that takes a chemical equation string and returns True if balanced, False if not.

**Your Task:**
- Split the equation at the arrow symbol "→" to get reactants and products
- For each side, parse the molecules and their coefficients
- Count total atoms of each element on both sides
- Compare the totals

Example Equation Format: "2H2 + O2 → 2H2O"

**Hints:**

- Use equation.split("→") to separate reactants from products
- Use side.split(" + ") to separate individual molecules
- You'll need to handle coefficients (numbers in front of molecules)
- Reuse your count_atoms() function from Exercise 1!
- For molecules with coefficients like "2H2O", the coefficient applies to all atoms in that molecule

**Steps to Think Through:**

- How do you extract the coefficient from "2H2O"? (It's everything before the first letter)
- How do you multiply the atom counts by the coefficient?
- How do you combine counts from multiple molecules on the same side?


In [7]:
from typing import Dict

def parse_molecular_formula(formula: str) -> Dict[str, int]:
    atom_dict = dict()

    #iterate through characters i in formula
    i=0
    #define lenght of the formula
    length = len(formula)
    #iter through the charecters of the formula
    for i in range(length):
        #distinuish between letters and numbers
        if formula[i].isalpha():
            #check if the letter already exists in the dictionary, if it does, add 1 to the value of the letter in the dictionary
            if formula[i] in atom_dict:
                atom_dict[formula[i]] += 1
            #check if the letter is uppercase or lowercase
            elif formula[i].isupper():
                #insert the letter into the dictionary
                atom_dict[formula[i]] = 1
            #if letter is lowercase, add it to the previous letter to complete the atom name and replace the name of the atom in the dictionary
            else:
                atom_dict[formula[i-1]+formula[i]]=1
                del atom_dict[formula[i-1]]
        #else if the character is a number, alter the value of the letter before 
        elif formula[i].isdigit():
            #check if character before is a number or a letter
            if formula[i-1].isalpha():
                #check if the attributing atom is a single letter or a double letter
                if formula[i-1].islower():
                    atom_dict[formula[i-2]+formula[i-1]]=int(formula[i])
                #otherwise just update the numbering of the atom in the dictionary
                else:
                    atom_dict[formula[i-1]] = int(formula[i])
            else:
                #check if the character before is a double number or a single number
                if formula[i-1].isdigit():
                    #check if the attributing atom is a single letter or a double letter
                    if formula[i-2].islower():
                        #add the numbers together and update the value of the atom in the dictionary
                        atom_dict[formula[i-3]+formula[i-2]]=int(int(formula[i])+(10*int(formula[i-1])))
                    else:
                        atom_dict[formula[i-2]]=int(int(formula[i])+(10*int(formula[i-1])))


    return atom_dict
print(parse_molecular_formula("H2OH"))

def is_balanced(chem_eq: str) -> bool:
    # split the equation into reactants and protducts
    reactants, products = chem_eq.split('→')
    # Split the reactants and products into individual compounds
    reactants_compounds= reactants.split("+")
    reactants_compounds = [compound.strip() for compound in reactants_compounds]
    products_compounds= products.split("+")
    products_compounds = [compound.strip() for compound in products_compounds]

    #look for coefficients in the reactants compounds
    #create a dictionary to store the number of atoms in the reactants
    reactants_atoms = {}
    i=0
    for i in range(len(reactants_compounds)):
        
        if reactants_compounds[i][0].isdigit():
            #determine coefficient
            c_r = int(reactants_compounds[i][0])
            reactants_compounds[i] = reactants_compounds[i][1:]
        else:
            c_r = 1
        #count number of atoms in the reactants with function
        parse_compound_r = parse_molecular_formula(reactants_compounds[i])
        #muliply all the atoms in the formula by the coefficient
        parse_compound_coeff_r = {key: c_r * value for key, value in parse_compound_r.items()}

        #add the atoms in the reactants to the reactants_atoms dictionary
        for key, value in parse_compound_coeff_r.items():
            if key in reactants_atoms:
                #add the value if it already exists in the dictionary
                reactants_atoms[key] += value
            else:
                reactants_atoms[key] = value

    #do the same for the products
    products_atoms = {}
    for i in range(len(products_compounds)):
        if products_compounds[i][0].isdigit():
            c_p = int(products_compounds[i][0])
            products_compounds[i] = products_compounds[i][1:]
        else:
            c_p = 1
        parse_compound_p = parse_molecular_formula(products_compounds[i])
        parse_compound_coeff_p = {key: c_p * value for key, value in parse_compound_p.items()}

        for key, value in parse_compound_coeff_p.items():
            if key in products_atoms:
                products_atoms[key] += value
            else:
                products_atoms[key] = value
    print(reactants_atoms)
    print(products_atoms)

    #compare the two dictionaries to see if they are equal
    if (dict(sorted(reactants_atoms.items())) == dict(sorted(products_atoms.items()))):
        return True
    else: 
        return False



print(is_balanced("2H2 + O2 → 2H2O"))  # Example usage

{'H': 3, 'O': 1}
{'H': 4, 'O': 2}
{'H': 4, 'O': 2}
True


### Test Cases


In [3]:
# If the outcome is incorrect, the keyword assert will cause an error

print(is_balanced("2H2 + O2 → 2H2O"))
assert is_balanced("2H2 + O2 → 2H2O")

print(is_balanced("H2 + O2 → H2O"))
assert not is_balanced("H2 + O2 → H2O")

print(is_balanced("CH4 + 2O2 → CO2 + 2H2O"))
assert is_balanced("CH4 + 2O2 → CO2 + 2H2O")

print(is_balanced("N2 + H2 → NH3"))
assert not is_balanced("N2 + H2 → NH3")

True
False
True
False


## Exercise 3: Reaction Pathway Analyzer

**Objective:** Analyze multi-step reaction sequences to find synthesis pathways.

**Problem:** Write a program that determines what compounds can be synthesized from starting materials through a series of reactions, and finds a pathway to create a target compound.

**Your Task:**
Create a function called `find_synthesis_path()` that takes:

- A list of starting compounds
- A list of reaction equations
- A target compound to synthesize

The function should return either:

- A list of reaction steps that lead to the target compound, OR
- A message saying the target cannot be synthesized

**Hints:**

- Start by creating a set of "available" compounds (your starting materials)
- For each reaction, check if you have all the reactants needed
- If you can perform a reaction, add the products to your available compounds
- Keep track of which reactions you've used in what order
- Continue until you either find the target or can't make any new compounds

#### Computational Thinking:

**Decomposition:** Break this into smaller problems

- Parse reaction equations
- Track available compounds
- Check if reactions can be performed
- Record the synthesis pathway


**Pattern Recognition:** What patterns do you see?

- Each reaction has the same format: "reactants → products"
- You need to repeatedly check if new reactions become possible


**Algorithm Design:** What's your step-by-step approach?

- Initialize available compounds
- Loop: try each reaction, see if possible, update available compounds
- Stop when target is found or no progress is made


**Abstraction:** What functions might be helpful?

- A function to parse a single reaction equation
- A function to check if a reaction is possible with current compounds
- A function to update available compounds after a reaction



In [9]:

from typing import List

#function to parse the equation and list reactants atoms and products atoms (derived from the is_balanced function)
def parse_equation(chem_eq: str) -> tuple[dict[str, int], dict[str, int]]:

    reactants, products = chem_eq.split("→")
    reactants_compounds = reactants.split("+")
    reactants_compounds = [compound.strip() for compound in reactants_compounds]

    products_compounds = products.split("+")
    products_compounds = [compound.strip() for compound in products_compounds]

    reactants_dict = {}
    products_dict = {}

    # reactants
    for compound in reactants_compounds:
        if compound[0].isdigit():
            coefficient = int(compound[0])
            compound = compound[1:]
        else:
            coefficient = 1

        reactants_dict[compound] = coefficient

    # products
    for compound in products_compounds:
        if compound[0].isdigit():
            coefficient = int(compound[0])
            compound = compound[1:]
        else:
            coefficient = 1

        products_dict[compound] = coefficient

    return reactants_dict, products_dict


print(parse_equation("2H2 + O2 → 2H2O"))

#function to asess if reaction is possible 

def reaction_possible(available_reactants: Dict[str, int], chem_eq: str) -> bool:
    reactants_dict = parse_equation(chem_eq)[0]
    for compound, count in reactants_dict.items():
        if compound not in available_reactants: #or available_reactants[compound] >= count: 
            return False
    return True

print(reaction_possible({'H2': 2, 'O2': 1}, "2H2 + O2 → 2H2O"))
        
#function that updates the list of available reactants if a new product is formed. 
def compound_update (available_reactants:Dict[str,int], chem_eq:str) -> Dict[str,int]:
    available_reactans_update= available_reactants.copy()
    #if reaction is possible, add the products to the available reactants.
    if reaction_possible(available_reactants,chem_eq) ==True: 
        for product, count in parse_equation(chem_eq)[1].items():
            available_reactans_update[product]=count
    #if reaction is not possible, keep available reactants as they are. 
    else: 
        available_reactans_update=available_reactants.copy()
    return available_reactans_update

print(compound_update({'H2': 2, 'O2': 1}, "2H2 + O2 → 2H2O"))



def find_synthesis_path(starting_materials: List[str], reactions: List[str], target: str) -> List[str] | str:
    #change starting materials list to a dictionary: 
    starting_m= starting_materials.copy()
    available_reactants={}
    m=0
    for m in range(len(starting_m)):
        if starting_m[m][0].isdigit():
            coefficient=int(starting_m[m][0])
            starting_m[m]=starting_m[m][1:]
        else: 
            coefficient=1 #if not defined, we have "infinite" amount of starting material
        available_reactants[starting_m[m]]=coefficient
    #return available_reactants
    #start an empty list of reaction tracking
    track_reactions=[]
    #set a maximum of loop repeats. otherwise the reaction pathway is not found 
    i=0
    for i in range(len(reactions)):

        #loop through all reactions
    
        if target not in available_reactants.keys():
            #loop through the reactions 
            for reaction in reactions:
                #check if reaction i is already listed in track_reactions. If reaction possible add to track_reactions
                if reaction not in track_reactions and reaction_possible(available_reactants,reaction)==True:
                    available_reactants= compound_update(available_reactants,reaction)
                    track_reactions.append(reaction)
                else:
                    continue
        else: 
            return track_reactions
    else:
        return "target cannot be synthetized"


##last two tests cannot be solved with this code...

({'H2': 2, 'O2': 1}, {'H2O': 2})
True
{'H2': 2, 'O2': 1, 'H2O': 2}


### Test Cases

In [10]:
# Basic Multi-Step Synthesis

starting_materials = ["H2", "O2", "N2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3", 
    "NH3 + H2O → NH4OH"
]
target = "NH4OH"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find a path using H2 + N2 → NH3, then NH3 + H2O → NH4OH

['2H2 + O2 → 2H2O', 'N2 + 3H2 → 2NH3', 'NH3 + H2O → NH4OH']


In [11]:
# Direct Synthesis (One Step)

starting_materials = ["Na", "Cl2"]
reactions = [
    "2Na + Cl2 → 2NaCl",
    "NaCl + H2O → NaOH + HCl"
]
target = "NaCl"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find direct synthesis in one step

['2Na + Cl2 → 2NaCl']


In [12]:
# Impossible Synthesis

starting_materials = ["H2", "O2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3"  # But we don't have N2!
]
target = "NH3"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should return message that NH3 cannot be synthesized

target cannot be synthetized


In [13]:
# Multiple Pathway Options
starting_materials = ["C", "H2", "O2", "H2O"]
reactions = [
    "C + O2 → CO2",
    "2H2 + O2 → 2H2O",
    "CO2 + H2O → H2CO3",
    "C + 2H2 → CH4",
    "CH4 + 2O2 → CO2 + 2H2O"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find one of multiple possible paths to CO2

['C + O2 → CO2', '2H2 + O2 → 2H2O', 'CO2 + H2O → H2CO3', 'C + 2H2 → CH4', 'CH4 + 2O2 → CO2 + 2H2O']


In [23]:
# Long Chain Synthesis
starting_materials = ["Fe", "O2", "C"]
reactions = [
    "4Fe + 3O2 → 2Fe2O3",
    "2C + O2 → 2CO",
    "Fe2O3 + 3CO → 2Fe + 3CO2",
    "CO2 + C → 2CO"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find the multi-step path involving iron oxide formation and reduction

['4Fe + 3O2 → 2Fe2O3', '2C + O2 → 2CO', 'Fe2O3 + 3CO → 2Fe + 3CO2', 'CO2 + C → 2CO']
